### Introduction to Rationality Calibration in Canonical Games

Welcome to "Notebook 4: Rationality Calibration in Canonical Games." This notebook isn't just about playing games; it's a carefully designed scientific experiment aimed at understanding how decision-making policies, both simple algorithmic ones and advanced artificial intelligence like Large Language Models (LLMs), behave in strategic situations. Our core objective is to investigate a fundamental question: **Does a decision policy consistently align with established game-theoretic solution concepts, specifically Pure Nash Equilibria, across familiar 2x2 games and even when those games are presented in different ways?**

#### The Big Picture: Why Game Theory?

Game theory provides a powerful framework for analyzing strategic interactions where the outcome for each participant depends on the actions of all participants. Concepts like **Nash Equilibrium** are central to understanding rational behavior in such scenarios. A Pure Nash Equilibrium is a state where no player can improve their outcome by unilaterally changing their action, assuming the other players' actions remain constant. In essence, it's a stable outcome that a rational actor would likely choose.

#### Our Approach: Testing Decision Policies

In this notebook, we evaluate various "policies" – systems or algorithms that make decisions. We examine two main types:

1.  **`synthetic_policy`**: This is a simulated, rule-based decision-maker. It incorporates elements of self-interest (maximizing its own payoff), social considerations (maximizing combined payoffs), and some controlled randomness. It serves as a valuable baseline to understand how specific, adjustable parameters influence decision-making and consistency with equilibrium.
2.  **`llm_policy_adapter`**: This is where we bring advanced AI into the picture. We integrate a Large Language Model (like Claude Haiku in your case) as a decision-maker. The `llm_policy_adapter` acts as a crucial bridge: it translates the mathematical structure of a game (its payoff matrix) into a clear, natural language prompt for the LLM. The LLM then processes this information and chooses an action, which the adapter parses back into a structured format we can analyze. This allows us to rigorously test whether the LLM's responses align with game-theoretic rationality.

#### The Games We Play: Canonical 2x2 Scenarios

We utilize classic 2x2 normal-form games, such as:
*   **Prisoner's Dilemma**: Highlights the tension between individual rationality and collective well-being.
*   **Stag Hunt**: Explores coordination and trust.
*   **Chicken**: A game of brinkmanship and risk.
*   **Coordination Game**: Focuses purely on mutual benefit through aligned actions.

For each game, we precisely define its **payoff matrix**, which details the rewards each player receives for every possible combination of actions. We then identify all its **Pure Nash Equilibria** – these are our benchmarks for 'equilibrium-consistent' decisions.

#### Beyond Simple Consistency: Representation Control

A truly robust claim of rationality should hold irrespective of how the problem is superficially presented. Therefore, a critical aspect of this notebook is **representation control**. We test the policies not only with numerical payoff matrices but also with different action labels (e.g., "neutral" labels like A/B, "social" labels like cooperate/defect, or "anonymous" labels like option_17/option_42). The goal is to determine if the semantic framing of actions influences a policy's decision-making, or if its adherence to equilibrium concepts remains consistent across equivalent representations.

#### The Null Hypothesis and What We Hope to Learn

Our formal starting point is the **null hypothesis**: "Observed choices are not more equilibrium-consistent than the declared stochastic baseline and do not transfer across representations." If our experiments allow us to *reject* this null hypothesis, it suggests that the tested policy exhibits a degree of consistency with fundamental game-theoretic principles. Conversely, if we cannot reject it, the policy's decisions might be indistinguishable from random chance or are heavily influenced by superficial framing.

#### Important Boundaries: What This Notebook *Cannot* Establish

It's crucial to understand the limitations of this study. This is a **safe, synthetic, mechanism-oriented experiment**. It does *not* claim to reproduce real-world incidents, nor does it establish a general property of large language models beyond the specific conditions tested. It cannot certify alignment, safety, consciousness, or broad rationality. All results are conditional on the stated rules, parameters, random seeds, and specific model used. This notebook aims to illustrate a specific mechanism or reject a narrow null hypothesis within a controlled, synthetic environment, providing foundational insights without overstating general implications.

By carefully calibrating decision policies against these well-understood game-theoretic concepts, we gain a clearer, more granular understanding of their strategic capabilities and limitations.

In [19]:
pip install anthropic -q

In [8]:
import anthropic
from google.colab import userdata

ANTHROPIC_API_KEY = userdata.get('ANTHROPIC_API_KEY')
client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

print("Anthropic client initialized.")

Anthropic client initialized.


In [22]:
import json, math, random, sys, platform
from dataclasses import dataclass, asdict
from itertools import combinations, product
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 762
rng = np.random.default_rng(SEED)
random.seed(SEED)
plt.style.use("seaborn-v0_8-whitegrid")

MANIFEST = {
    "seed": SEED,
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "mode": "safe_synthetic",
}
print(json.dumps(MANIFEST, indent=2))

{
  "seed": 762,
  "python": "3.13.15",
  "platform": "Linux-6.6.122+-x86_64-with-glibc2.35",
  "mode": "safe_synthetic"
}


## Canonical games as calibration, not certification

We test a synthetic decision policy in Prisoner's Dilemma, Stag Hunt, Chicken, and a coordination game. Passing these games would show consistency with the specified solution concepts. It would not certify alignment, general rationality, or safety.

In [23]:
games = {
    "Prisoners_Dilemma": np.array([[[3,3],[0,5]], [[5,0],[1,1]]]),
    "Stag_Hunt": np.array([[[4,4],[0,3]], [[3,0],[2,2]]]),
    "Chicken": np.array([[[-5,-5],[1,4]], [[4,1],[0,0]]]),
    "Coordination": np.array([[[3,3],[0,0]], [[0,0],[2,2]]]),
}

def pure_nash(payoffs):
    equilibria = []
    for i,j in product(range(2), repeat=2):
        u1,u2 = payoffs[i,j]
        if u1 >= payoffs[1-i,j,0] and u2 >= payoffs[i,1-j,1]:
            equilibria.append((i,j))
    return equilibria

pd.DataFrame({name: {"pure_nash": pure_nash(p)} for name,p in games.items()}).T

,pure_nash
Prisoners_Dilemma,"[(1, 1)]"
Stag_Hunt,"[(0, 0), (1, 1)]"
Chicken,"[(0, 1), (1, 0)]"
Coordination,"[(0, 0), (1, 1)]"


In [24]:
def synthetic_policy(payoffs, temperature=.20, social_weight=.10, seed=SEED):
    local = np.random.default_rng(seed)
    scores = []
    for i,j in product(range(2), repeat=2):
        own = payoffs[i,j,0]
        social = payoffs[i,j].sum()
        scores.append(((i,j), own + social_weight*social + local.gumbel(0, temperature)))
    return max(scores, key=lambda x:x[1])[0]

rows=[]
for game,p in games.items():
    eq=set(pure_nash(p))
    for social_weight in [0,.1,.3,.6]:
        for rep in range(300):
            action=synthetic_policy(p,social_weight=social_weight,seed=SEED+rep)
            rows.append((game,social_weight,action,action in eq))
calibration=pd.DataFrame(rows,columns=["game","social_weight","action","equilibrium_consistent"])
calibration.groupby(["game","social_weight"]).equilibrium_consistent.mean().unstack().round(3)

social_weight,0.0,0.1,0.3,0.6
game,,,,
Chicken,1.000,1.0,1.0,1.0
Coordination,1.000,1.0,1.0,1.0
Prisoners_Dilemma,0.000,0.0,0.0,0.0
Stag_Hunt,0.997,1.0,1.0,1.0


## Representation control

A rationality claim should transfer across equivalent presentations. We compare numerical matrices with relabeled action names while keeping payoffs identical. The synthetic policy is invariant by construction; an optional future LLM treatment can test whether semantic framing changes behavior.

In [25]:
labels = {
    "neutral": {0:"A",1:"B"},
    "social": {0:"cooperate",1:"defect"},
    "anonymous": {0:"option_17",1:"option_42"},
}
records=[]
p=games["Prisoners_Dilemma"]
for frame,mapping in labels.items():
    for rep in range(100):
        a=synthetic_policy(p,seed=SEED+rep)
        records.append({"frame":frame,"raw_action":a,"display_action":(mapping[a[0]],mapping[a[1]])})
pd.DataFrame(records).groupby(["frame","raw_action"]).size().unstack(fill_value=0)

raw_action,"(1, 0)"
frame,
anonymous,100
neutral,100
social,100


## Optional future LLM adapter

The notebook deliberately makes no paid API call. A later study may replace `synthetic_policy` with a versioned model adapter. The adapter must log the model name, system prompt, user prompt, temperature, raw response, parsing rule, retry policy, and cost. Results must remain model- and configuration-specific.

In [26]:
def llm_policy_adapter(*args, **kwargs):
    raise NotImplementedError(
        "No external model is called in the safe default notebook. "
        "Implement only after specifying provenance, cost controls, and a reproducibility manifest."
    )

print("Safe default: synthetic policy only; no API key required.")

Safe default: synthetic policy only; no API key required.


In [27]:
import re
# Modified llm_policy_adapter to use Anthropic Claude Haiku
def llm_policy_adapter(game_payoffs, model_name="claude-haiku-4-5-20251001", temperature=0.7, client_instance=None):
    if client_instance is None:
        raise ValueError("Anthropic client instance must be provided to llm_policy_adapter.")

    # Format the payoff matrix for the prompt
    p00 = game_payoffs[0, 0]
    p01 = game_payoffs[0, 1]
    p10 = game_payoffs[1, 0]
    p11 = game_payoffs[1, 1]

    prompt_template = f"""
    You are playing a 2x2 normal form game. Your task is to choose an action for Player 0 and Player 1.
    The payoff matrix for (Player 0's choice, Player 1's choice) -> (Player 0's payoff, Player 1's payoff) is as follows:

    - If Player 0 chooses 0 and Player 1 chooses 0, payoffs are: Player 0 gets {p00[0]}, Player 1 gets {p00[1]}
    - If Player 0 chooses 0 and Player 1 chooses 1, payoffs are: Player 0 gets {p01[0]}, Player 1 gets {p01[1]}
    - If Player 0 chooses 1 and Player 1 chooses 0, payoffs are: Player 0 gets {p10[0]}, Player 1 gets {p10[1]}
    - If Player 0 chooses 1 and Player 1 chooses 1, payoffs are: Player 0 gets {p11[0]}, Player 1 gets {p11[1]}

    Player 0's available actions are: 0 or 1.
    Player 1's available actions are: 0 or 1.

    Respond ONLY with a Python tuple representing the chosen joint actions for Player 0 and Player 1.
    The format MUST be exactly: (P0_action, P1_action)
    Where P0_action is 0 or 1, and P1_action is 0 or 1.
    DO NOT include any other text, explanation, or punctuation.
    Your response should look like: (0,1)
    """

    messages = [
        {"role": "user", "content": prompt_template.strip()}
    ]

    try:
        response = client_instance.messages.create(
            model=model_name,
            max_tokens=50, # Increased max_tokens to accommodate potential verbosity, though prompt aims to prevent it
            messages=messages
        )
        content = response.content[0].text.strip()
        print(f"DEBUG: LLM raw content: '{content}'") # Debug print

        # Use regex to find a tuple-like string like (X,Y)
        match = re.search(r'\((\d),\s*(\d)\)', content)

        if match:
            parsed_actions = (int(match.group(1)), int(match.group(2)))
        else:
            # If regex fails, the format is incorrect; do not use eval.
            print(f"Warning: LLM returned unparseable action (no tuple found by regex): '{content}'. Returning (0,0).")
            return (0,0) # Default if parsing fails

        if isinstance(parsed_actions, tuple) and len(parsed_actions) == 2 and \
           all(isinstance(a, int) and a in [0, 1] for a in parsed_actions):
            return parsed_actions
        else:
            print(f"Warning: LLM returned invalid tuple values or format: '{parsed_actions}'. Returning (0,0).")
            return (0,0) # Default if values are out of range

    except Exception as e:
        print(f"Error calling LLM or parsing response: {e}. Returning (0,0).")
        return (0,0) # Default on error

print("LLM policy adapter ready.")

LLM policy adapter ready.


## LLM Policy Calibration

In [28]:
llm_rows = []
llm_model_name = "claude-haiku-4-5-20251001"
llm_temperature = 0.7 # Can be varied if needed
num_repetitions = 1 # LLM calls can be expensive, so starting with one repetition

print(f"Running LLM calibration for model: {llm_model_name} with {num_repetitions} repetition(s) per game.")

for game_name, p in games.items():
    for rep in range(num_repetitions):
        # Calling the llm_policy_adapter for each game
        # Pass the initialized client instance
        action_llm = llm_policy_adapter(p, model_name=llm_model_name, temperature=llm_temperature, client_instance=client)
        eq = set(pure_nash(p))
        llm_rows.append((game_name, llm_model_name, action_llm, action_llm in eq))

llm_calibration = pd.DataFrame(llm_rows, columns=["game", "model", "action", "equilibrium_consistent"])

print("\nLLM Calibration Results:")
display(llm_calibration.groupby(["game", "model"]).equilibrium_consistent.mean().unstack(fill_value=0).round(3))

Running LLM calibration for model: claude-haiku-4-5-20251001 with 1 repetition(s) per game.
DEBUG: LLM raw content: '(0,0)'
DEBUG: LLM raw content: '(0,0)'
DEBUG: LLM raw content: '(1,0)'
DEBUG: LLM raw content: '(0,0)'

LLM Calibration Results:


model,claude-haiku-4-5-20251001
game,
Chicken,1.0
Coordination,1.0
Prisoners_Dilemma,0.0
Stag_Hunt,1.0


## Strongest permissible conclusion

Canonical games calibrate whether a decision procedure is consistent with specified equilibrium concepts under controlled representations. They do not establish a universal or human-equivalent rationality standard.

## Evidentiary boundary

Interpret every numerical result as conditional on the stated rules, parameters, seed, and model class. A result may illustrate a mechanism or reject a null hypothesis inside this synthetic environment. It does not establish intention, consciousness, deception, alignment, or universal LLM behavior.

### Conclusion: Insights from Policy Calibration

This notebook set out to calibrate decision-making policies against game-theoretic solution concepts, specifically Pure Nash Equilibria, across canonical 2x2 games. We evaluated both a `synthetic_policy` and an `llm_policy_adapter` (using Claude Haiku) for their consistency with these rational benchmarks.

**1. Synthetic Policy Performance:**

Our `synthetic_policy`, which incorporates adjustable `social_weight` and stochasticity, demonstrated strong equilibrium consistency in several games:

*   For **Chicken**, **Coordination**, and **Stag Hunt** games, the synthetic policy consistently (near 100%) chose actions that were Pure Nash Equilibria across different social weighting parameters. This indicates that its internal logic, even with some consideration for collective payoffs, generally aligns with individual rationality in these contexts.
*   However, for the **Prisoner's Dilemma**, the synthetic policy consistently chose actions *not* in equilibrium (0% consistency). This is a crucial observation. The Pure Nash Equilibrium in Prisoner's Dilemma is often the 'defect-defect' outcome, which is individually rational but socially suboptimal. Given that our synthetic policy has a `social_weight` component, it appears to favor the cooperative outcome (which leads to higher collective payoffs but is not a Nash Equilibrium) over the purely self-interested Nash outcome. This highlights how an explicit social preference can steer a policy away from a Nash equilibrium.

**2. LLM Policy (Claude Haiku) Performance:**

The `llm_policy_adapter` provided us with an initial look at how `claude-haiku-4-5-20251001` performs:

*   **Equilibrium Consistency:** Claude Haiku successfully identified and chose Pure Nash Equilibrium actions for the **Stag Hunt**, **Chicken**, and **Coordination** games in our single-shot trials. This suggests that for these specific games and their textual representations, the LLM's decision-making aligns with game-theoretic rationality.
*   **Divergence in Prisoner's Dilemma:** Mirroring the behavior of our synthetic policy, the LLM also chose an action that was *not* a Pure Nash Equilibrium for the **Prisoner's Dilemma**. It opted for the cooperative `(0,0)` action, which is often considered socially optimal but deviates from the individually rational Nash Equilibrium of `(1,1)` (defect-defect). This suggests a potential inherent bias towards cooperation or social welfare in the LLM's responses, similar to what we observed when the `synthetic_policy` had a positive `social_weight`.

**3. Key Takeaways and Future Directions:**

Both policies exhibited a notable divergence from Pure Nash Equilibrium in the Prisoner's Dilemma, potentially driven by a preference for cooperative or higher-sum outcomes. This initial calibration demonstrates the utility of this framework for dissecting the decision-making tendencies of both rule-based algorithms and complex LLMs.

It is important to emphasize that the LLM results are based on a very limited number of repetitions (one per game). To draw more robust conclusions, further experiments with more repetitions, varying `temperature` parameters, and exploring different prompt formulations (especially for representation control) would be necessary. Nevertheless, this exercise provides valuable initial insights into the game-theoretic 'rationality' of our selected LLM, highlighting both its alignment and its specific divergences from traditional economic rationality in strategic settings.